In [1]:
##IMPLEMENTACIÓN DE UNA RNN BÁSICA
#Construimos una RNN de forma interna, sin usar las funciones que ya están implementadas en TensorFlow.
#Será una red con una capa de 5 neuronas recurrentes, con la función de activación tanh.
#Asumo que se ejecutará solo para dos pasos de tiempo, tomando en cada paso entradas de tamaño 3.
import tensorflow as tf
n_inputs=3
n_neurons=5
Wx = tf.Variable(tf.random.normal([n_inputs, n_neurons], dtype=tf.float32))
Wy = tf.Variable(tf.random.normal([n_neurons, n_neurons], dtype=tf.float32))
b = tf.Variable(tf.zeros([1, n_neurons], dtype=tf.float32))

#Entradas 
X0 = tf.random.normal([4, n_inputs])  # tamaño batch = 4, solo para probar, con datos reales se pondrían tales datos.
X1 = tf.random.normal([4, n_inputs])

#Paso hacia delante
Y0 = tf.tanh(tf.matmul(X0, Wx) + b)
Y1 = tf.tanh(tf.matmul(Y0, Wy) + tf.matmul(X1, Wx) + b)


In [2]:
#Output de la red en t=0 para todas las neuronas y ejemplos en el minibatch
print(Y0.numpy())   

[[-0.89111483  0.9877741  -0.89113355  0.82758254 -0.7862613 ]
 [-0.7398352   0.9342553  -0.7443187   0.5975982  -0.05515228]
 [ 0.55039716  0.34195685  0.17976823  0.09643079  0.9721762 ]
 [ 0.5182221  -0.9955131   0.8180204  -0.91752726 -0.12965119]]


In [3]:
#Output de la red en t=1 para todas las neuronas y ejemplos en el minibatch
print(Y1.numpy())   

[[-0.9889814  -0.34422687 -0.7438328   0.9787763   0.9979282 ]
 [-0.9545778   0.9328787   0.3836546   0.98858035  0.9818982 ]
 [-0.08711728  0.99866635  0.9860031   0.39124146  0.73343307]
 [ 0.9331581  -0.6701589  -0.980569   -0.9841218  -0.6865139 ]]


In [ ]:
#Ahora veamos cómo crear el mismo modelo usando las operaciones de RNN de TensorFlow.
n_inputs=3
n_neurons=5
n_steps=2
#Creamos el modelo 
model = tf.keras.models.Sequential([
    tf.keras.layers.SimpleRNN(n_neurons, return_sequences=True, input_shape=[n_steps, n_inputs])
])
#return_sequences=True devuelve los outputs de cada paso de tiempo (Y0, Y1...); si no, solo da el resultado del último paso.
#El tamaño del batch se maneja internamente, es como si recibiera (None,n_steps,n_inputs).


/opt/anaconda3/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [5]:
#Datos de entrada (X_batch): los genero aleatoriamente (batch_size=4).
X_batch = tf.random.normal([4, n_steps, n_inputs])

#Por último ejecutamos:
outputs_val = model(X_batch)

print("Forma de la salida:", outputs_val.shape) # [4, 2, 5] -> [Batch, Pasos, Neuronas]
print(outputs_val)

Forma de la salida: (4, 2, 5)
tf.Tensor(
[[[-0.88435924 -0.4498386  -0.7865666  -0.6029227   0.5280661 ]
  [ 0.8557572   0.15520881 -0.15964341 -0.8360709   0.670852  ]]

 [[ 0.47911528  0.01581232  0.3193231   0.41054195 -0.4286922 ]
  [-0.49529117 -0.15481469  0.19470865  0.7183685  -0.27137163]]

 [[ 0.50982     0.01371268  0.46768203  0.22385536 -0.4147538 ]
  [-0.42985922 -0.03936009  0.3226416   0.5718969  -0.12586644]]

 [[ 0.12470474  0.7639438   0.71989703 -0.9285442   0.904532  ]
  [ 0.9511142   0.56189996  0.20658424  0.8949387  -0.9858434 ]]], shape=(4, 2, 5), dtype=float32)


In [ ]:
#Hasta ahora hemos gestionado inputs con tamaño fijo (2 pasos cada uno), pero puede que las secuencias de inputs tengan longitudes variables (como las frases).
import numpy as np
n_inputs = 3
n_neurons = 5
n_stepsmax = 2 # Longitud máxima
X_batch = np.array([         #Lo hago así porque si los genero aleatoriamente no me va a salir uno de ceros entero.
    [[0, 1, 2], [9, 8, 7]], # Instancia 0: completa
    [[3, 4, 5], [0, 0, 0]], # Instancia 1: solo 1 paso real
    [[6, 7, 8], [6, 5, 4]], # Instancia 2: completa
    [[9, 0, 1], [3, 2, 1]], # Instancia 3: completa
], dtype="float32")

#Para mejorar esto, definimos el modelo con una capa de Masking
model = tf.keras.models.Sequential([
    #Le decimos que si el vector es [0,0,0], es relleno
    tf.keras.layers.Masking(mask_value=0.0, input_shape=(n_stepsmax, n_inputs)),
    tf.keras.layers.SimpleRNN(n_neurons, return_sequences=True)
])

outputs = model(X_batch)
print(outputs)

#Si miramos los resultados de la instancia 1, vemos que la fila del paso 0 y la del paso 1 son idénticas. 
#Esto es para que si después usamos return_sequences=False, el modelo tenga disponible el último valor real calculado.

tf.Tensor(
[[[-0.9602477  -0.8816549  -0.8213305   0.69955367  0.45889995]
  [-0.999716    0.45132247 -0.99999964  0.9999162   0.42748934]]

 [[-0.9992384  -0.95460963 -0.99977034  0.9862852   0.73819876]
  [-0.9992384  -0.95460963 -0.99977034  0.9862852   0.73819876]]

 [[-0.9999857  -0.9829972  -0.9999997   0.9994607   0.88472354]
  [-0.98235387  0.8688604  -0.99965864  0.99904925  0.11558148]]

 [[ 0.9965538   0.9961664  -0.9997146   0.99996907 -0.51273406]
  [ 0.63548034  0.86380285 -0.9981787   0.84139866 -0.86128646]]], shape=(4, 2, 5), dtype=float32)


/opt/anaconda3/lib/python3.13/site-packages/keras/src/layers/core/masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
